# PDF 문서 기반 RAG 실습

블로그 글 [RAG(Retrieval-augmented generation)](https://seojuny.dev/what-is-rag) 의 실습 코드다. 가상의 회사 **구름테크**의 취업규칙 PDF(`data/sample-취업규칙.pdf`)를 두고, 직원이 규정을 직접 찾지 않아도 궁금한 걸 물으면 답해주는 사내 Q&A 를 만든다.

프레임워크(LangChain 등) 없이 `pypdf` + `openai` + `chromadb` 만으로, 텍스트가 벡터가 되고 검색되는 과정을 직접 본다.

흐름은 두 단계다.

- **인덱싱** (1~3): 질문이 오기 전에 문서를 벡터 DB 에 저장
- **답변 생성** (4~5): 질문이 들어올 때마다 검색해서 답하기

## 0. 환경 세팅 (먼저 터미널에서 한 번)

노트북 셀을 실행하려면 **커널이 먼저 있어야 한다.** 아래를 **터미널에서** 한 번 실행해 환경·커널을 만든 뒤, 이 노트북에서 그 커널을 고른다.

### 1) 파이썬 버전 확인
**Python 3.10 이상**이면 된다. (3.14 에서도 동작 확인)

```bash
python --version        # 3.10 이상인지 확인
```

너무 낮으면(3.9 이하) `pyenv` 로 올린다. (`pyenv` 가 없으면 `brew install pyenv` 후)

```bash
pyenv install 3.12.7    # 예시. 3.10 이상이면 아무 버전이나 OK
```

### 2) pipenv 설치 + 라이브러리 한 번에
`pipenv` 가 없으면 먼저 설치한다. `pipenv install` 이 가상환경 생성과 패키지 설치(`Pipfile` 기준)를 한 번에 처리한다.

```bash
pip install --user pipenv      # 없을 때만. (또는: brew install pipenv)

cd rag-pdf-practice
pipenv install                 # Pipfile.lock 의 정확한 버전으로 설치돼 누구나 같은 환경이 된다
```

### 3) Jupyter 커널 등록
VS Code / Cursor 의 커널 목록에 **RAG PDF Practice** 가 보이게 한다.

```bash
pipenv run python -m ipykernel install --user --name rag-pdf-practice --display-name "RAG PDF Practice"
```

### 4) 모델 선택 — OpenAI 또는 로컬 Ollama

**(A) OpenAI 를 쓰려면** API 키를 넣는다.

```bash
cp .env.example .env           # .env 를 열어 OPENAI_API_KEY 를 채운다 (sk-... )
```

**(B) 로컬 모델(무료)을 쓰려면** `.env` 의 `OPENAI_API_KEY` 를 **비워 두고**, [Ollama](https://ollama.com) 를 설치해 모델을 받아 둔다. 키가 없으면 `get_client()` 가 자동으로 Ollama 로 붙는다.

```bash
ollama pull qwen2.5            # 답변 생성 모델
ollama pull nomic-embed-text  # 임베딩 모델
# ollama 가 백그라운드로 떠 있어야 한다 (http://localhost:11434)
```

세팅이 끝나면 오른쪽 위(VS Code)/상단(Jupyter)에서 커널을 **RAG PDF Practice** 로 고르고, 아래 셀부터 차례로 실행한다.

In [ ]:
# 준비 — LLM 클라이언트 만들기 (OpenAI 또는 로컬 Ollama 자동 선택)
#   커널이 'RAG PDF Practice' 로 선택돼 있어야 한다 (0번 환경 세팅 참고)
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY 를 환경변수로 읽는다

def get_client():
    """OPENAI_API_KEY 가 있으면 OpenAI, 없으면 로컬 Ollama 를 사용한다.
    반환: (client, 답변 모델, 임베딩 모델) — 임베딩과 답변 둘 다 같은 client 로 호출한다."""
    if os.getenv("OPENAI_API_KEY", "").strip().startswith("sk-"):
        client = OpenAI()
        chat_model = "gpt-4o-mini"
        embedding_model = "text-embedding-3-small"
        print("✅ OpenAI 연결 — 답변:", chat_model, "| 임베딩:", embedding_model)
    else:
        # Ollama 의 OpenAI 호환 엔드포인트. 먼저 모델을 받아 둬야 한다:
        #   ollama pull qwen2.5 && ollama pull nomic-embed-text
        # (임베딩을 리스트로 한 번에 보내므로 배치 입력을 지원하는 최신 Ollama 권장)
        client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        chat_model = "qwen2.5"
        embedding_model = "nomic-embed-text"
        print("✅ Ollama 로컬 연결 — 답변:", chat_model, "| 임베딩:", embedding_model)
    return client, chat_model, embedding_model

client, CHAT_MODEL, EMBEDDING_MODEL = get_client()

## ⚙️ CONFIG — 여기 값만 바꿔가며 실험한다

이 셀의 값을 바꾸고 아래를 다시 실행하면 답이 어떻게 달라지는지 볼 수 있다.

| 값 | 의미 |
|---|---|
| `CHUNK_SIZE` | 청크 하나의 글자 수. 크면 한 조각에 여러 내용이 섞여 검색이 흐려지고, 작으면 문맥이 끊긴다 |
| `OVERLAP` | 청크 사이 겹치는 글자 수. 경계에서 문장이 잘리는 걸 막는다 |
| `N_RESULTS` | 검색해서 가져올 청크 수(top-k). 적으면 근거가 빠지고, 많으면 잡음이 섞인다 |
| `SYSTEM_PROMPT` | 모델에게 주는 답변 규칙. 환각 억제·출처 인용 강제 등을 여기서 조절 |
| `TEMPERATURE` | 0에 가까울수록 문서에 충실, 높을수록 자유로움. 사실 질의는 0~0.3 권장 |

`CHUNK_SIZE`·`OVERLAP` 을 바꾸면 벡터 DB 내용 자체가 달라지므로 **2(청킹) → 3(인덱싱)** 셀을 다시 실행해야 하고, `N_RESULTS`·`SYSTEM_PROMPT`·`TEMPERATURE` 는 질문할 때만 쓰이므로 **질문 셀만** 다시 실행하면 된다.

In [ ]:
# ── 문서 ──────────────────────────────────
PDF_PATH = "data/sample-취업규칙.pdf"

# 모델(답변·임베딩)은 위 '준비' 셀의 get_client() 가 백엔드(OpenAI/Ollama)에 맞춰 자동으로 정한다.
# 다른 모델로 바꾸고 싶으면 get_client() 안의 모델명을 수정한다.

# ── 청킹 (바꾸면 재인덱싱 필요) ──────────────
CHUNK_SIZE = 200   # 샘플 문서가 작아 200 정도면 조항 단위로 검색이 구분된다. 키우면서 검색이 어떻게 흐려지는지 봐도 좋다
OVERLAP = 40

# ── 검색·생성 (바꾸면 재인덱싱 불필요) ────────
N_RESULTS = 3
TEMPERATURE = 0
SYSTEM_PROMPT = "다음 문서를 근거로 질문에 답하세요. 문서에 없는 내용은 모른다고 답하세요."

## 1. 문서에서 텍스트 추출

PDF 에서 페이지별 텍스트를 뽑아 하나로 잇는다. 표·이미지가 많은 PDF 는 추출 품질이 떨어질 수 있으니, 이 결과를 한 번 눈으로 확인해 두는 게 좋다.

In [ ]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
text = "\n".join(page.extract_text() or "" for page in reader.pages)

print(f"총 {len(text)}자 추출\n")
print(text[:300], "...")

## 2. 텍스트 청킹

문서를 통째로 임베딩하면 한 벡터에 여러 내용이 뭉뚱그려져 검색이 흐려진다. 그래서 작은 조각으로 나눈다. 다만 **무조건 작을수록 좋은 건 아니고**, `CHUNK_SIZE` 는 한쪽으로 치우치면 둘 다 문제가 된다.

- **너무 크면**: 한 조각에 여러 주제가 섞여, 질문과 일부만 관련 있어도 통째로 딸려 오고 검색이 흐려진다.
- **너무 작으면**: 문장·문맥이 잘려, 정작 답에 필요한 정보가 조각 경계에서 끊긴다.

질문 한 건이 필요로 하는 정보량에 맞는 적당한 크기를 찾는 게 핵심이다. `OVERLAP` 은 조각 경계에서 끊기는 문맥을 다음 조각에 조금 겹쳐 메운다. 여기서는 글자 수 기준으로 단순하게 자른다. (`CHUNK_SIZE`, `OVERLAP` 을 바꿨다면 이 셀을 다시 실행)

In [ ]:
def chunk_text(text, chunk_size, overlap):
    assert overlap < chunk_size, "OVERLAP 은 CHUNK_SIZE 보다 작아야 합니다 (안 그러면 무한 반복)"
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

print(f"청크 {len(chunks)}개 (chunk_size={CHUNK_SIZE}, overlap={OVERLAP})\n")

# 전체 청크를 확인한다. 청크가 길면 앞부분만 잘라서 ... 로 표시
PREVIEW = 120
for i, c in enumerate(chunks):
    head = c[:PREVIEW].replace("\n", " ")
    more = " ..." if len(c) > PREVIEW else ""
    print(f"[{i}] ({len(c)}자) {head}{more}")

## 3. 임베딩과 벡터 DB 저장

각 청크를 임베딩 모델에 넣어 의미를 담은 숫자 벡터로 바꾸고, 원문과 함께 벡터 DB(Chroma)에 저장한다. 검색으로 찾은 벡터의 원문을 꺼내 모델에 넘겨야 하므로 `documents` 에 원문도 같이 넣는다.

청킹 설정을 바꿔 다시 실행할 때 이전 청크가 남지 않도록, 컬렉션을 매번 새로 만든다.

In [ ]:
import chromadb

def build_index(chunks):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks)
    embeddings = [d.embedding for d in resp.data]

    db = chromadb.PersistentClient(path="./chroma_db")
    try:
        db.delete_collection("docs")  # 재인덱싱 시 이전 내용 비우기
    except Exception:
        pass
    # hnsw:space="cosine" → 검색 거리를 코사인 거리로 잰다 (유사도 = 1 - 거리)
    collection = db.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=[f"chunk-{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks,
    )
    return collection

collection = build_index(chunks)
print(f"인덱싱 완료 — {collection.count()}개 청크 저장")

## 4~5. 질문 임베딩 · 유사도 검색 · 답변 생성

질문도 **같은 임베딩 모델**로 벡터로 바꾼 뒤 가장 가까운 청크 `N_RESULTS` 개를 찾고, 그 청크를 컨텍스트로 삼아 모델에게 답을 생성하게 한다.

- 답하는 규칙(지시)은 `system` 에, 검색한 컨텍스트와 질문은 `user` 에 나눠 넣는다.
- `SYSTEM_PROMPT` 의 "문서에 없으면 모른다고" 한 줄이 환각을 억제한다.
- `show_context=True` 로 호출하면 검색된 청크와 **코사인 유사도 점수**(1에 가까울수록 질문과 비슷)를 함께 보여준다. 어떤 근거로 답했는지, 검색이 잘 됐는지 눈으로 확인할 수 있다.

`ask()` 는 호출할 때마다 `CONFIG` 의 `N_RESULTS`·`SYSTEM_PROMPT`·`TEMPERATURE` 를 읽으므로, 이 값들만 바꿨다면 재인덱싱 없이 이 셀과 아래 질문 셀만 다시 실행하면 된다.

In [ ]:
def ask(question, show_context=False):
    q_emb = client.embeddings.create(
        model=EMBEDDING_MODEL, input=question
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[q_emb],
        n_results=N_RESULTS,
        include=["documents", "distances"],  # 거리도 함께 받아온다
    )
    retrieved = results["documents"][0]
    distances = results["distances"][0]
    similarities = [1 - d for d in distances]  # 코사인 공간: 유사도 = 1 - 거리 (1에 가까울수록 비슷)

    if show_context:
        print(f"=== 검색된 청크 {len(retrieved)}개 (유사도 높은 순) ===")
        for i, (c, s) in enumerate(zip(retrieved, similarities)):
            print(f"[{i}] 유사도 {s:.3f} | {c[:90].strip()} ...")
        print("=" * 40, "\n")

    context = "\n\n".join(retrieved)
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"[문서]\n{context}\n\n[질문]\n{question}"},
        ],
        temperature=TEMPERATURE,
    )
    return resp.choices[0].message.content

In [ ]:
question = "연차 휴가는 며칠까지 쓸 수 있어?"
print(ask(question, show_context=True))

## RAG 있을 때 vs 없을 때

같은 질문을 RAG 없이 그냥 물으면, 모델이 PDF 내용을 모르니 일반론을 답하거나 사실과 다른 내용을 지어낸다. 둘을 나란히 비교해 본다.

In [ ]:
def ask_without_rag(question):
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=TEMPERATURE,
    )
    return resp.choices[0].message.content

print("🚫 RAG 없이:\n", ask_without_rag(question), "\n")
print("✅ RAG 있음:\n", ask(question))

## 🧪 직접 실험해 보기

위 `CONFIG` 셀의 값을 바꿔가며 답이 어떻게 달라지는지 본다.

**재인덱싱이 필요 없는 변경** (`N_RESULTS`, `SYSTEM_PROMPT`, `TEMPERATURE`)
→ `CONFIG` 셀 실행 → `ask(...)` 셀만 다시 실행

**재인덱싱이 필요한 변경** (`CHUNK_SIZE`, `OVERLAP`)
→ `CONFIG` 셀 실행 → **2(청킹) → 3(인덱싱)** 셀 다시 실행 → `ask(...)`

아래 셀은 여러 질문을 한 번에 던져 보는 용도다. 실험거리 몇 가지:

- `N_RESULTS` 를 1 로 줄이면 근거가 빠져 "모른다"가 나오기 쉽다. 5~8 로 늘리면?
- `SYSTEM_PROMPT` 에 `"답변 끝에 근거가 된 조항 번호를 (제O조) 형식으로 붙이세요."` 를 더해 출처를 인용하게 해 보자.
- `CHUNK_SIZE` 를 100 으로 줄이면 조항이 잘려 검색이 어떻게 흔들리는가?
- 문서에 없는 질문("회사 주차장은 몇 층이야?")을 던지면 환각을 억제하는지 본다.

In [ ]:
questions = [
    "배우자가 출산하면 며칠 쉴 수 있어?",
    "재택근무는 주 며칠까지 가능해?",
    "월급은 언제 들어와?",
    "회사 주차장은 몇 층이야?",  # 문서에 없는 질문 — 모른다고 답해야 한다
]

for q in questions:
    print(f"Q. {q}")
    print(f"A. {ask(q)}\n")